# Application 2: Autonomous AI Agent System using Llama

This notebook implements a LangChain agent system powered by a local Llama model (via Ollama).

## What this satisfies
- Integration of Llama model with LangChain agents
- At least two tools/functions (implemented: search, calculator, file reader)
- Multi-step reasoning and decision-making workflow
- Dynamic tool selection based on user input
- Memory for context retention
- Structured output for agent responses
- Error handling for tool failures/invalid inputs

## User Guide

### 1) Setup Instructions
1. Install Ollama from: https://ollama.com/download
2. Pull a Llama model (example):
   ```bash
   ollama pull llama3.1
   ```
3. Ensure Ollama server is running:
   ```bash
   ollama serve
   ```

### 2) Required Dependencies
Install in your environment (compatible with LangChain 1.2.x):
```bash
pip install -U langchain langchain-core langchain-community langgraph pydantic
```

### 3) How to Run
1. Run all cells from top to bottom.
2. Use `run_agent_query(...)` with your prompt.
3. Inspect structured JSON-like output for final answer, reasoning steps, tools used, and errors.

### 4) Example Usage Scenarios
- Ask a factual question requiring search over local knowledge.
- Ask a math question requiring calculator use.
- Ask to read a local text file and summarize its content.
- Ask a follow-up question to verify memory context retention.

In [3]:
# Core imports
import json
import ast
import operator as op
from pathlib import Path
from typing import List

from pydantic import BaseModel, Field, ValidationError

from langchain_community.chat_models import ChatOllama
from langchain_core.messages import HumanMessage
from langchain_core.tools import tool
from langgraph.checkpoint.memory import MemorySaver
from langgraph.prebuilt import create_react_agent

In [4]:
# Structured response schema for post-validation
class AgentResponse(BaseModel):
    final_answer: str = Field(description="Primary answer to the user")
    reasoning_steps: List[str] = Field(default_factory=list, description="Short reasoning trace")
    tools_used: List[str] = Field(default_factory=list, description="Tools used by the agent")
    status: str = Field(default="success", description="success or error")
    error: str = Field(default="", description="Error message if any")

In [6]:
# Small local knowledge base for the search tool
LOCAL_KB = [
    "LangChain supports tool-using agents with planning and action loops.",
    "Llama models can be served locally with Ollama and integrated through ChatOllama.",
    "Agent memory can retain prior user context across turns for better continuity.",
    "Structured outputs can be validated with Pydantic schemas for robust downstream use.",
]

# Safe expression evaluator for calculator inputs
_SAFE_OPERATORS = {
    ast.Add: op.add,
    ast.Sub: op.sub,
    ast.Mult: op.mul,
    ast.Div: op.truediv,
    ast.Pow: op.pow,
    ast.USub: op.neg,
}

def _safe_eval(expr: str):
    def _eval(node):
        if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
            return node.value
        if isinstance(node, ast.BinOp) and type(node.op) in _SAFE_OPERATORS:
            return _SAFE_OPERATORS[type(node.op)](_eval(node.left), _eval(node.right))
        if isinstance(node, ast.UnaryOp) and type(node.op) in _SAFE_OPERATORS:
            return _SAFE_OPERATORS[type(node.op)](_eval(node.operand))
        raise ValueError("Unsupported expression. Use basic arithmetic only.")

    parsed = ast.parse(expr, mode="eval")
    return _eval(parsed.body)

@tool
def local_search(query: str) -> str:
    """Search a small local knowledge base and return matched lines."""
    try:
        q = query.lower().strip()
        matches = [line for line in LOCAL_KB if q in line.lower()]
        if not matches:
            return "No relevant local knowledge found."
        return "\n".join(matches[:3])
    except Exception as exc:
        return f"TOOL_ERROR(local_search): {exc}"

@tool
def calculator(expression: str) -> str:
    """Evaluate a safe arithmetic expression, e.g. (23*7)+5/2."""
    try:
        result = _safe_eval(expression)
        return str(result)
    except Exception as exc:
        return f"TOOL_ERROR(calculator): {exc}"

@tool
def file_reader(path: str) -> str:
    """Read text from a local file path relative to current project root."""
    try:
        root = Path.cwd().resolve()
        candidate = (root / path).resolve()

        # Prevent directory traversal outside project root
        if root not in candidate.parents and candidate != root:
            return "TOOL_ERROR(file_reader): Access outside project root is not allowed."

        if not candidate.exists() or not candidate.is_file():
            return "TOOL_ERROR(file_reader): File not found."

        content = candidate.read_text(encoding="utf-8", errors="ignore")
        if len(content) > 3000:
            return content[:3000] + "\n... [truncated]"
        return content
    except Exception as exc:
        return f"TOOL_ERROR(file_reader): {exc}"

In [7]:
# Llama + LangChain agent configuration (LangChain 1.2.x style via LangGraph)
MODEL_NAME = "llama3.1"  # Change if your local model tag differs

llm = ChatOllama(model=MODEL_NAME, temperature=0)
tools = [local_search, calculator, file_reader]

SYSTEM_PROMPT = (
    "You are an autonomous AI assistant that can reason in multiple steps and use tools when needed. "
    "Choose tools dynamically based on user intent. "
    "Always return your final result in STRICT JSON with keys: "
    "final_answer (string), reasoning_steps (list of short strings), tools_used (list of strings), "
    "status (success/error), error (string). "
    "If a tool fails, continue when possible and report the failure in error."
)

checkpointer = MemorySaver()
agent_executor = create_react_agent(
    model=llm,
    tools=tools,
    prompt=SYSTEM_PROMPT,
    checkpointer=checkpointer,
)

# Thread id keeps conversation context across calls
THREAD_ID = "lab8-agent-session"

/tmp/ipykernel_96035/938176781.py:4: LangChainDeprecationWarning: The class `ChatOllama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import ChatOllama``.
  llm = ChatOllama(model=MODEL_NAME, temperature=0)
/tmp/ipykernel_96035/938176781.py:17: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent_executor = create_react_agent(


NotImplementedError: 

In [ ]:
def _coerce_to_structured_output(raw_output: str, inferred_tools: List[str] | None = None) -> AgentResponse:
    """Validate/repair the model output into the AgentResponse schema."""
    inferred_tools = inferred_tools or []
    try:
        parsed = json.loads(raw_output)
        if inferred_tools and not parsed.get("tools_used"):
            parsed["tools_used"] = inferred_tools
        return AgentResponse(**parsed)
    except (json.JSONDecodeError, ValidationError):
        # Fallback if model output is not valid JSON
        return AgentResponse(
            final_answer=raw_output.strip(),
            reasoning_steps=["Output was not valid JSON; wrapped raw output."],
            tools_used=inferred_tools,
            status="success",
            error="",
        )

def _extract_last_text_and_tools(messages) -> tuple[str, List[str]]:
    """Extract final assistant text and tool names from LangGraph message state."""
    raw = ""
    tools_used: List[str] = []

    for msg in messages:
        msg_type = getattr(msg, "type", "")
        if msg_type == "tool":
            name = getattr(msg, "name", "tool")
            tools_used.append(str(name))

    if messages:
        last = messages[-1]
        content = getattr(last, "content", "")
        if isinstance(content, list):
            parts = [part.get("text", "") for part in content if isinstance(part, dict)]
            raw = "\n".join([p for p in parts if p]).strip()
        else:
            raw = str(content).strip()

    return raw, tools_used

def run_agent_query(user_query: str) -> dict:
    """Run one query through the agent with robust error handling and structured output."""
    if not isinstance(user_query, str) or not user_query.strip():
        return AgentResponse(
            final_answer="",
            reasoning_steps=[],
            tools_used=[],
            status="error",
            error="Invalid input: user_query must be a non-empty string.",
        ).model_dump()

    try:
        result = agent_executor.invoke(
            {"messages": [HumanMessage(content=user_query)]},
            config={"configurable": {"thread_id": THREAD_ID}},
        )
        messages = result.get("messages", [])
        raw, inferred_tools = _extract_last_text_and_tools(messages)
        structured = _coerce_to_structured_output(raw, inferred_tools=inferred_tools)
        return structured.model_dump()
    except Exception as exc:
        return AgentResponse(
            final_answer="",
            reasoning_steps=[],
            tools_used=[],
            status="error",
            error=f"Agent execution failed: {exc}",
        ).model_dump()

## Example Runs
Run each example to observe tool selection and multi-step behavior.

In [ ]:
# Example 1: search + reasoning
response_1 = run_agent_query("What does local knowledge say about LangChain and Llama integration?")
print(json.dumps(response_1, indent=2))

# Example 2: calculator
response_2 = run_agent_query("Calculate (17*9) + 25/5 and explain shortly.")
print(json.dumps(response_2, indent=2))

# Example 3: file reader
response_3 = run_agent_query("Read README.md and summarize the project in 3 bullet points.")
print(json.dumps(response_3, indent=2))

# Example 4: memory follow-up
response_4 = run_agent_query("Based on our previous result, what was the math answer you gave me?")
print(json.dumps(response_4, indent=2))

## Notes
- If model/tool output is not perfectly JSON, the system wraps raw output into schema safely.
- Tool failures are captured and surfaced through the structured `error` field.
- You can expand `LOCAL_KB` or replace it with a real retrieval pipeline for stronger search.